In [ ]:
import geopandas as gpd
import pandas as pd
import os

In [ ]:
# Load spatial data.
sal_2011_gdf = gpd.read_file("2011_Census/ea_sal_kzn_gp/ea_sal_kzn_gp.shp")
ward_2023_gdf = gpd.read_file("2023_Census/SA_Wards2020/SA_Wards2020.shp")

# Filter ward_2023_gdf to KwaZulu-Natal and Gauteng only.
ward_2023_gdf = ward_2023_gdf[ward_2023_gdf["Province"].isin(["KwaZulu-Natal", "Gauteng"])]
print(f"Filtered wards: {len(ward_2023_gdf)} records (KwaZulu-Natal and Gauteng only).")

# Ensure projected coordinate reference system for accurate area math.
sal_2011_gdf = sal_2011_gdf.to_crs("EPSG:32735")
ward_2023_gdf = ward_2023_gdf.to_crs("EPSG:32735")

# Calculate original sal area before intersection.
sal_2011_gdf["orig_sal_area"] = sal_2011_gdf.geometry.area

# Perform geometric overlay to cut polygons at intersections.
# This creates the one to many relationship physically.
areal_weighted_gdf = gpd.overlay(sal_2011_gdf, ward_2023_gdf, how = "intersection")

# Calculate new area for the cut slivers.
areal_weighted_gdf["new_sliver_area"] = areal_weighted_gdf.geometry.area

# Calculate the areal weight ratio.
areal_weighted_gdf["weight_ratio"] = areal_weighted_gdf["new_sliver_area"] / areal_weighted_gdf["orig_sal_area"]

# Convert population to numeric type and apply the weight to the 2011 population.
areal_weighted_gdf["weighted_pop_2011"] = pd.to_numeric(areal_weighted_gdf["population"], errors = "coerce") * areal_weighted_gdf["weight_ratio"]

# Verification of successful execution.
print(f"Result: {len(areal_weighted_gdf)} intersection polygons.")

In [ ]:
# Export the areal weighted result to a new shapefile.
output_dir = "areal_weighted_output"
os.makedirs(output_dir, exist_ok = True)

output_path = os.path.join(output_dir, "areal_weighted_kzn_gp.shp")
areal_weighted_gdf.to_file(output_path)

print(f"Shapefile exported to: {output_path}")
print(f"Total features exported: {len(areal_weighted_gdf)}.")